# Konvolúciós neurális hálózat

LeCun, Yann, Bernhard Boser, John Denker, et al. “Handwritten Digit Recognition with a Back-Propagation Network.” Advances in Neural Information Processing Systems 2 (1989). https://proceedings.neurips.cc/paper/1989/hash/53c3bce66e43be4f209556518c2fcb54-Abstract.html.

In [17]:
# !pip install torchinfo

In [18]:
import torch
from torch import nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torchinfo import summary

In [19]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [20]:
num_workers = 2
batch_size = 256
epochs = 10

In [21]:
data_transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [ ]:
train_data = datasets.MNIST(
    root="data", train=True, download=True, transform=data_transform
)
test_data = datasets.MNIST(
    root="data", train=False, download=True, transform=data_transform
)

In [23]:
train_data.classes

['0 - zero',
 '1 - one',
 '2 - two',
 '3 - three',
 '4 - four',
 '5 - five',
 '6 - six',
 '7 - seven',
 '8 - eight',
 '9 - nine']

In [24]:
train_dataloader = torch.utils.data.DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
)

test_dataloader = torch.utils.data.DataLoader(
    test_data,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)

## Modell

A **konvolúciós neurális hálózat** (convolutional neural network, CNN ) egy olyan neurális hálózat, amely legalább egy rétegében - az általános mátrix multiplikáció helyett - konvolúciót$^1$  használ. Lásd [itt](https://www.deeplearningbook.org/contents/convnets.html).

Általában az alábbi rétegek valamely kombinációja található a hálózatban: konvolúciós réteg (convolutional layer), összevonó (pooling) réteg, sorosító (flatten) réteg és teljesen összekapcsolt (feed-forward) réteg.

A **konvolúció** a matematikában diszkrét, illetve folytonos $x, w$ függvények esetén a

$$
\begin{aligned}
& (x * w)(t)=\sum_{\tau=-\infty}^{\infty} x(\tau) w(t-\tau) \\
& (x * w)(t)=\int x(\tau) w(t-\tau) d \tau
\end{aligned}
$$

összegzéssel, illetve integrállal definiált függvények értendők. A CNN nevezéktanában a konvolúció első argumentuma ( $x$ ) a bemenet, a második argumentum ( $w$ ) a **kernel** (vagy szűrő), illetve a kimenet a **jellemzőleképezés** (feature map).

A gépi tanulásban a bemenet egy mátrix, a kernel egy tanítható mátrix (a konvolúciós réteg paraméterei)$^{2}$.

A konvolúció gyakran egynél több tengely mentén kerül alkalmazásra. Például, ha a bemenet egy kétdimenziós $I$ kép, akkor célszerű kétdimenziós $K$ kernelt használni, amelynek számítási módját az alábbi egyenlet szemlélteti:

$$S(i, j)=(I * K)(i, j)=\sum_m \sum_n I(m, n) K(i-m, j-n).$$

Mielőtt továbbmennénk, nézzünk egy konkrét példát. Vegyük az alábbi $\mathbf{I}$ bemenetet és $\mathbf{K}$ kernelt:


$$
\begin{aligned}
&\mathbf{I}=\left[\begin{array}{lll}
0 & 1 & 2 \\
3 & 4 & 5 \\
6 & 7 & 8
\end{array}\right] \quad \mathbf{K}=\left[\begin{array}{ll}
0 & 1 \\
2 & 3
\end{array}\right]
\end{aligned}
$$

Az egyenlet alapján a diszkrét konvolúció az alábbi módon kerül kiszámításra:

$$
\begin{aligned}
& \mathbf{S}(0,0)=0 \cdot 0+1 \cdot 1+3 \cdot 2+4 \cdot 3=19, \\
& \mathbf{S}(0,1)=1 \cdot 0+2 \cdot 1+4 \cdot 2+5 \cdot 3=25, \\
& \mathbf{S}(1,0)=3 \cdot 0+4 \cdot 1+6 \cdot 2+7 \cdot 3=37, \\
& \mathbf{S}(1,1)=4 \cdot 0+5 \cdot 1+7 \cdot 2+8 \cdot 3=43 .
\end{aligned}
$$

Az eredményként kapott jellemzőleképezés pedig:

$$
\mathbf{S}=\left[\begin{array}{ll}
19 & 25 \\
37 & 43
\end{array}\right]
$$

A modern mélytanulási keretrendszerek a számítási és implementációs egyszerűség miatt a **kereszt-korrelációt** (cross-correlation) használnak, és ezt konvolúciónak nevezik.

A két eljárás közötti különbség mindössze annyi, hogy a diszkrét konvolúcióhoz a $\mathbf{K}$ kernelt horizontális és vertikális tükrözni kell. A gyakorlatban a tanítható paraméterek miatt a tükrözés elhagyása nem okoz problémát, hiszen a hálózat úgyis megtanulja a megfelelő mintázatot.

A jellemzőleképezésre a modell által megtanult reprezentációkként (vagy jellemzőkként) is lehet gondolni, amire rendszerint egy nemlineáris aktivációs függvényt is alkalmazunk, mielőtt a következő réteg feldolgozná.

A **csatornák** (channels) a jellemzőleképezések mélységi dimenzióját adják meg, ahol minden egyes csatorna egy-egy jellemzőről hordoz információt. Egy színes kép esetén például három bemeneti csatornáról beszélünk (RGB), míg a konvolúciós rétegek által előállított kimeneti csatornák száma a használt szűrők számával egyezik meg. Minden szűrő egy adott mintázatra vagy jellemzőre érzékeny, ezért a csatornák az egymással párhuzamosan tanult reprezentációkat kódolják. A csatornák dimenziója így a hálózat kapacitását is meghatározza: minél több csatorna áll rendelkezésre, annál gazdagabb jellemzőtér alakítható ki a bemenetből.

Az **összevonó réteg** az előző konvolúciós réteg alkalmazása által kapott reprezentáció méretét csökkenti azáltal, hogy felosztja részekre vagy szeletekre. Utána, az ún. **lépésköz**nek (stride) megfelelóen végigmegy a szeleten és minden egyes lépésben alkalmazza az átlag vagy maximum számítás műveletét$^3$. Ez a paraméterek számának és a komputáció mennyiségének csökkenését eredményezi, hozzájárulva a gyorsabb tanuláshoz. Gyakori - de nem kötelező - az egymást követő konvolúciós rétegek közé helyezett összevonó réteg.

A **sorosító réteg** nem tartalmaz tanítható paramétert és feladata a bemeneti többdimenziós mátrix egydimenziós vektorrá alakítása, hogy lehetővé váljon a teljesen összekapcsolt réteg számára a további feldolgozás.

Végül, a **teljesen összekapcsolt lineáris réteg**ek a konvolúciós rétegek, valamint az összevonó és sorosító műveletek után helyezkednek el.

Ezek a rétegek a korábbi rétegek által kinyert jellemzőket a kimeneti osztályokhoz rendelhető diszkrét valószínűségi eloszlássá képezik le$^4$. A gyakorlatban a lineáris rétegek között aktivációs függvény (pl. ReLU) és dropout művelet$^5$ is alkalmazható a nemlinearitás, illetve a túlilleszkedés csökkentése érdekében.

Például két teljesen összekapcsolt réteg kimenete a következőképpen számolható:

$$
\begin{aligned}
& \mathbf{h}=\operatorname{Dropout}_p\left(\max \left(0, \mathbf{x} \mathbf{W}_1+\mathbf{b}_1\right)\right), \\
& \mathbf{y}=\mathbf{h} \mathbf{W}_2+\mathbf{b}_2,
\end{aligned}
$$

ahol $\mathbf{W}_1 \in \mathbb{R}^{a \times b}, \mathbf{W}_2 \in \mathbb{R}^{b \times c}$ a súlymátrixok, $\mathbf{b}_1 \in \mathbb{R}^b, \mathbf{b}_2 \in \mathbb{R}^c$ az eltolásvektorok, $c$ az osztályok száma, $\max (0, \cdot)$ a ReLU aktivációs függvény, és $\operatorname{Dropout}_p(\cdot)$ a dropout művelet $p$ valószínűséggel.

Megjegyzés.

$^1$ A jelfeldolgozás nevezéktanában ezt a műveletet nem konvolúciónak, hanem keresztkorrelációnak nevezik, a mélytanulásban azonban a konvolúció kifejezés elterjedt.

$^2$ Egy $m \times n$-es mátrixot (ahol $m$ és $n$ pozitív egész számok) fel lehet fogni egy olyan $\mathbf{A}$ függvényként, amely minden $(i, j)$ párhoz, amelyre $1 \leq i \leq m$ és $1 \leq j \leq n$, hozzárendel egy elemet. Továbbá, mivel a bemenet és a kernel minden egyes elemét külön kell tárolni, általában azt feltételezik, hogy ezek a függvények mindenhol nulla értéket vesznek fel, kivéve azon a véges pontok halmazán, ahol az értékeik ténylegesen eltárolásra kerülnek.

$^3$ Bővebben lásd: https://developers.google.com/machine-learning/glossary#pooling

$^4$ Valószínűségről akkor beszélünk, ha softmax függvény (vagy más hasonló függvény) is alkalmazásra kerül ebben a lépésben, ha nem, akkor a kimenetek ún. logitok lesznek (amikre egész egyszerűen érdemes "nem-normalizált" számokként gondolni).

$^5$ A dropout művelet a tanítás során véletlenszerűen kinullázza a bemeneti (aktivációs) tenzor elemeinek egy részét $p$ valószínűséggel. Bővebben lásd: https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html

In [25]:
class CnnModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)

        self.pool = nn.MaxPool2d(2)

        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

        self.dropout = nn.Dropout(0.25)

    def forward(self, x: torch.Tensor):

        x = self.conv1(x)
        x = F.relu(x)

        x = self.conv2(x)
        x = F.relu(x)

        x = self.pool(x)
        x = torch.flatten(x, 1)

        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [26]:
model = CnnModel().to(device)

In [27]:
summary(model)

Layer (type:depth-idx)                   Param #
CnnModel                                 --
├─Conv2d: 1-1                            320
├─Conv2d: 1-2                            18,496
├─MaxPool2d: 1-3                         --
├─Linear: 1-4                            1,179,776
├─Linear: 1-5                            1,290
├─Dropout: 1-6                           --
Total params: 1,199,882
Trainable params: 1,199,882
Non-trainable params: 0

In [28]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.7)

In [29]:
def train(dataloader):

    model.train()

    train_loss, train_acc = 0, 0

    for X, y in dataloader:

        X, y = X.to(device), y.to(device)

        y_pred = model(X)

        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item()/len(y_pred)

    return train_loss / len(dataloader), train_acc / len(dataloader)

In [30]:
@torch.inference_mode()
def eval(dataloader):

    model.eval()

    test_loss, test_acc = 0.0, 0.0

    for X, y in dataloader:

        X, y = X.to(device), y.to(device)

        logits = model(X)

        loss = loss_fn(logits, y)
        test_loss += loss.item()

        test_pred_labels = logits.argmax(dim=1) == y
        test_acc  += test_pred_labels.float().mean().item()

    return test_loss / len(dataloader), test_acc / len(dataloader)

In [31]:
best_acc = 0.0

for epoch in range(epochs):
    train_loss, train_acc = train(train_dataloader)
    test_loss, test_acc = eval(test_dataloader)

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), "best_mnist_cnn.pt")

    print(
        f"Epoch: {epoch+1:2d} | "
        f"Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f} | "
        f"Test loss: {test_loss:.4f} | Test acc: {test_acc:.4f} | "
    )

Epoch:  1 | Train loss: 0.3309 | Train acc: 0.9111 | Test loss: 0.0741 | Test acc: 0.9771 | 
Epoch:  2 | Train loss: 0.0941 | Train acc: 0.9710 | Test loss: 0.0550 | Test acc: 0.9838 | 
Epoch:  3 | Train loss: 0.0739 | Train acc: 0.9777 | Test loss: 0.0490 | Test acc: 0.9837 | 
Epoch:  4 | Train loss: 0.0629 | Train acc: 0.9805 | Test loss: 0.0450 | Test acc: 0.9872 | 
Epoch:  5 | Train loss: 0.0580 | Train acc: 0.9820 | Test loss: 0.0488 | Test acc: 0.9865 | 
Epoch:  6 | Train loss: 0.0494 | Train acc: 0.9842 | Test loss: 0.0493 | Test acc: 0.9858 | 
Epoch:  7 | Train loss: 0.0461 | Train acc: 0.9855 | Test loss: 0.0479 | Test acc: 0.9868 | 
Epoch:  8 | Train loss: 0.0441 | Train acc: 0.9861 | Test loss: 0.0567 | Test acc: 0.9861 | 
Epoch:  9 | Train loss: 0.0421 | Train acc: 0.9864 | Test loss: 0.0618 | Test acc: 0.9856 | 
Epoch: 10 | Train loss: 0.0441 | Train acc: 0.9861 | Test loss: 0.0541 | Test acc: 0.9848 | 
